In [63]:
import shutil
from pathlib import Path 

import cv2
import dlib 
import IPython
import numpy as np
import pandas as pd 
from tqdm import tqdm 
from retinaface import RetinaFace

2024-10-21 17:52:07.100643: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [114]:
def parse_vector(vector):
    # return dlib.vector([float(x.strip()) for x in vector[1:-2].split(' ') if x])
    return dlib.vector([float(x) for x in vector.split('\n')])

In [4]:
def show_image(cap, row):
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    _, ret = cv2.imencode('.jpg', frame)
    i = IPython.display.Image(data=ret)
    IPython.display.display(i)

In [119]:
src = '../data/seinfeld_find_faces_queue_single.csv'
df = pd.read_csv(src, index_col=0)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,mouth_right_y,mouth_left_x,mouth_left_y,confidence,face_num,frame_num,encoding,img_width,img_height,filepath
23,0.345,0.229,0.377,0.314,0.351,0.264,0.365,0.266,0.355,0.282,...,0.296,0.362,0.298,0.998,0,72,-0.103766\n0.0299908\n0.027249\n0.0222682\n-0....,1920,1080,/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEB...
30,0.356,0.243,0.388,0.324,0.375,0.275,0.385,0.275,0.385,0.292,...,0.306,0.381,0.306,0.999,0,96,-0.0588099\n0.0478108\n0.0342819\n-0.0359363\n...,1920,1080,/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEB...
34,0.422,0.224,0.455,0.310,0.442,0.258,0.452,0.260,0.450,0.276,...,0.289,0.447,0.291,0.999,0,120,-0.107796\n0.0707972\n0.0720443\n-0.0647952\n-...,1920,1080,/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEB...
39,0.451,0.192,0.485,0.285,0.467,0.228,0.481,0.227,0.479,0.241,...,0.262,0.481,0.261,1.000,0,144,-0.117145\n0.109224\n0.12342\n-0.0726684\n-0.0...,1920,1080,/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEB...
44,0.499,0.194,0.534,0.289,0.505,0.234,0.521,0.234,0.511,0.253,...,0.269,0.520,0.269,0.999,0,168,-0.0999805\n0.0543887\n-0.0190107\n0.00427168\...,1920,1080,/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEB...


In [120]:
filepath = df.at[0, 'filepath']
print(filepath)
cap = cv2.VideoCapture(filepath)

/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEBRip.x265-RARBG/Seinfeld.S01.1080p.WEBRip.x265-RARBG/Seinfeld.S01E04.1080p.WEBRip.x265-RARBG.mp4


In [121]:
encodings = df['encoding'].map(parse_vector).tolist()
labels = dlib.chinese_whispers_clustering(encodings, 0.5)
df = df.assign(label=labels)
counts = df['label'].value_counts()
df_cnt = df.merge(counts,
              how='left',
              on='label')
top = df_cnt[df_cnt['count'] >= 15]
top.groupby('count').max()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,mouth_left_x,mouth_left_y,confidence,face_num,frame_num,encoding,img_width,img_height,filepath,label
count,,,,,,,,,,,,,,,,,,,,,
2393,0.973,0.757,0.998,0.999,0.99,0.849,1.019,0.856,1.001,0.96,...,1.011,1.044,1.0,7,32904,0.040789\n0.0457496\n0.00628241\n-0.0202079\n-...,1920,1080,/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEB...,0


In [54]:
dst_dir = Path('../data/seinfeld')
if not dst_dir.exists():
    Path.mkdir(dst_dir)
for idx, row in tqdm(top.iterrows(), total=df.shape[0]):
    dst = dst_dir.joinpath(str(row['label']))
    if not dst.exists():
        Path.mkdir(dst)
    fp = dst.joinpath(f'{row["frame_num"]}_{row["face_num"]}.png')
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    face = frame[row['y1']:row['y2'], row['x1']:row['x2']]
    cv2.imwrite(str(fp), face)

 16%|█▌        | 396/2522 [01:46<09:30,  3.73it/s]


KeyboardInterrupt: 

In [55]:
homeland = pd.read_csv('/home/amos/programs/FacesOfHomeland/data/faces.csv', index_col=0)
homeland.head()

,imdb_id,frame_num,face_num,x1,y1,x2,y2,img_height,img_width,area,pct_of_frame,season,episode,encoding,filename,character,cast_id,cluster
0,1811020,192,0,798,196,916,314,1080,1920,13924,0.007,1,1,"[-0.1581144630908966, 0.11875572055578232, 0.0...",S01E01_192_0.png,NaN,NaN,-1.0
1,1811020,216,0,1068,389,1491,812,1080,1920,178929,0.086,1,1,"[-0.035924218595027924, 0.09460100531578064, 0...",S01E01_216_0.png,NaN,NaN,NaN
2,1811020,216,1,912,17,1205,311,1080,1920,86142,0.042,1,1,"[-0.10312943160533905, 0.1261938065290451, 0.0...",S01E01_216_1.png,NaN,NaN,NaN
3,1811020,240,0,1090,106,1384,400,1080,1920,86436,0.042,1,1,"[-0.18763160705566406, 0.08110883831977844, 0....",S01E01_240_0.png,NaN,NaN,NaN
4,1811020,240,1,674,522,968,816,1080,1920,86436,0.042,1,1,"[-0.11751651763916016, 0.1057925745844841, 0.0...",S01E01_240_1.png,NaN,NaN,NaN


In [60]:
dst_dir = Path('../data/homeland')
temp = homeland[homeland['cluster'].notna()]
if not dst_dir.exists():
    Path.mkdir(dst_dir)
for idx, row in tqdm(temp.iterrows(), total=temp.shape[0]):
    dst = dst_dir.joinpath(str(row['cluster']))
    if not dst.exists():
        Path.mkdir(dst)
    fp = dst.joinpath(f'{row["frame_num"]}_{row["face_num"]}.png')
    src = Path('/home/amos/programs/FacesOfHomeland/data/homeland_2011_1796960').joinpath(row['filename'])
    if not src.exists():
        continue
    shutil.copy(str(src), str(fp))

  0%|          | 0/246879 [00:00<?, ?it/s]

 40%|███▉      | 97689/246879 [01:33<02:22, 1048.88it/s]


KeyboardInterrupt: 

In [66]:
def format_prediction(prediction, shape):
    h, w = shape
    # Why am I converting to relative coordinates?
    x1 = prediction['facial_area'][0]
    y1 = prediction['facial_area'][1]
    x2 = prediction['facial_area'][2]
    y2 = prediction['facial_area'][3]
    datum = {'x1': x1,
                'y1': y1, 
                'x2': x2, 
                'y2': y2}
    for k, v in prediction['landmarks'].items():
        x, y = v 
        datum[f'{k}_x'] = round(x/w, 3) 
        datum[f'{k}_y'] = round(y/h, 3)
    datum['confidence'] = round(prediction['score'], 3)
    return datum

In [70]:
def process_image(face):
    rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (150, 150), interpolation=cv2.INTER_AREA)
    return np.array(resized, dtype=np.uint8)

In [68]:
model_path = '../data/dlib_face_recognition_resnet_model_v1.dat'
encoder = dlib.face_recognition_model_v1(str(model_path))

In [123]:
src = '/home/amos/media/tv/Seinfeld.S01-S09.1080p.WEBRip.x265-RARBG/Seinfeld.S01.1080p.WEBRip.x265-RARBG/Seinfeld.S01E04.1080p.WEBRip.x265-RARBG.mp4'
cap = cv2.VideoCapture(src)

frame_num = 0
data = []
framecount = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
pb = tqdm(total=framecount)
images = {}
while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break 
    elif frame_num % 24 == 0:
        faces = RetinaFace.detect_faces(frame)
        for num, (_, d) in enumerate(faces.items()):
            datum = format_prediction(d, frame.shape[:2])
            face = frame[datum['y1']:datum['y2'], datum['x1']:datum['x2']]
            img = process_image(face)
            encoding = encoder.compute_face_descriptor(img)
            datum['encoding'] = encoding
            datum['frame_num'] = frame_num 
            datum['face_num'] = num
            data.append(datum)
            images[f'{datum["frame_num"]}_{datum["face_num"]}'] = img
    frame_num += 1
    pb.update()

  1%|          | 394/33364 [00:06<09:44, 56.42it/s]


100%|██████████| 33364/33364 [09:49<00:00, 79.69it/s] 

In [124]:
df = pd.DataFrame(data)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,mouth_right_x,mouth_right_y,mouth_left_x,mouth_left_y,confidence,encoding,frame_num,face_num
0,664,247,725,340,0.351,0.264,0.365,0.266,0.355,0.281,0.352,0.296,0.363,0.297,0.998,"[-0.08072326332330704, 0.05089209973812103, 0....",72,0
1,685,261,745,351,0.376,0.276,0.385,0.276,0.385,0.292,0.374,0.305,0.381,0.306,0.999,"[-0.08707177639007568, 0.05739464983344078, 0....",96,0
2,812,241,876,337,0.442,0.258,0.452,0.260,0.451,0.277,0.439,0.289,0.447,0.292,0.999,"[-0.09564363956451416, 0.060665786266326904, 0...",120,0
3,866,206,932,308,0.467,0.228,0.481,0.227,0.480,0.241,0.471,0.262,0.481,0.261,1.000,"[-0.08593324571847916, 0.10685312747955322, 0....",144,0
4,960,210,1026,314,0.506,0.235,0.521,0.234,0.511,0.253,0.509,0.269,0.520,0.269,0.998,"[-0.1032012477517128, 0.059800125658512115, 0....",168,0


In [125]:
encodings = list(df['encoding'].values)
labels = dlib.chinese_whispers_clustering(encodings, 0.5)
temp_df = df.assign(label=labels)
counts = temp_df['label'].value_counts()
df_cnt = temp_df.merge(counts,
              how='left',
              on='label')
top = df_cnt[df_cnt['count'] >= 15]
print(top['label'].value_counts())
top

label
0    2522
Name: count, dtype: int64


,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,mouth_right_x,mouth_right_y,mouth_left_x,mouth_left_y,confidence,encoding,frame_num,face_num,label,count
0,664,247,725,340,0.351,0.264,0.365,0.266,0.355,0.281,0.352,0.296,0.363,0.297,0.998,"[-0.08072326332330704, 0.05089209973812103, 0....",72,0,0,2522
1,685,261,745,351,0.376,0.276,0.385,0.276,0.385,0.292,0.374,0.305,0.381,0.306,0.999,"[-0.08707177639007568, 0.05739464983344078, 0....",96,0,0,2522
2,812,241,876,337,0.442,0.258,0.452,0.260,0.451,0.277,0.439,0.289,0.447,0.292,0.999,"[-0.09564363956451416, 0.060665786266326904, 0...",120,0,0,2522
3,866,206,932,308,0.467,0.228,0.481,0.227,0.480,0.241,0.471,0.262,0.481,0.261,1.000,"[-0.08593324571847916, 0.10685312747955322, 0....",144,0,0,2522
4,960,210,1026,314,0.506,0.235,0.521,0.234,0.511,0.253,0.509,0.269,0.520,0.269,0.998,"[-0.1032012477517128, 0.059800125658512115, 0....",168,0,0,2522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2517,725,327,917,600,0.438,0.405,0.467,0.397,0.474,0.449,0.444,0.497,0.465,0.493,0.999,"[-0.048055995255708694, 0.04533916711807251, 0...",32808,0,0,2522
2518,725,317,913,599,0.436,0.406,0.466,0.399,0.472,0.450,0.443,0.500,0.465,0.495,0.999,"[-0.14031937718391418, 0.03320857509970665, 0....",32832,0,0,2522
2519,726,316,914,600,0.437,0.406,0.467,0.398,0.473,0.450,0.444,0.500,0.465,0.495,0.999,"[-0.13779179751873016, 0.036015234887599945, 0...",32856,0,0,2522
2520,724,313,916,600,0.439,0.409,0.466,0.396,0.474,0.452,0.444,0.499,0.465,0.491,1.000,"[-0.04892945662140846, 0.024306215345859528, 0...",32880,0,0,2522


In [108]:
cap = cv2.VideoCapture('../data/videos/shining_bat.mp4')
dst_dir = Path('../data/shining')
if not dst_dir.exists():
    Path.mkdir(dst_dir)
for idx, row in tqdm(top.iterrows(), total=df.shape[0]):
    dst = dst_dir.joinpath(str(row['label']))
    if not dst.exists():
        Path.mkdir(dst)
    fp = dst.joinpath(f'{row["frame_num"]}_{row["face_num"]}.png')
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    face = frame[row['y1']:row['y2'], row['x1']:row['x2']]
    cv2.imwrite(str(fp), face)

 92%|█████████▏| 282/308 [00:21<00:02, 12.82it/s]


In [110]:
shining = pd.read_csv('../data/shining.csv', index_col=0)
shining.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,mouth_right_y,mouth_left_x,mouth_left_y,confidence,face_num,frame_num,encoding,img_width,img_height,filepath
1,0.377,0.154,0.541,0.556,0.416,0.289,0.496,0.276,0.458,0.341,...,0.444,0.495,0.432,0.998,0,24,-0.11073\n0.0121753\n0.0814921\n-0.0949048\n-0...,1280,720,/home/amos/programs/CineFace/data/videos/shini...
4,0.403,0.181,0.570,0.583,0.444,0.318,0.526,0.312,0.487,0.390,...,0.482,0.522,0.477,0.996,0,48,-0.121618\n0.0306692\n0.101491\n-0.114291\n-0....,1280,720,/home/amos/programs/CineFace/data/videos/shini...
8,0.399,0.172,0.564,0.572,0.440,0.310,0.521,0.305,0.483,0.376,...,0.467,0.518,0.463,0.998,0,72,-0.140024\n0.0493508\n0.0980294\n-0.10492\n-0....,1280,720,/home/amos/programs/CineFace/data/videos/shini...
11,0.408,0.167,0.572,0.562,0.456,0.305,0.535,0.299,0.504,0.361,...,0.458,0.534,0.452,0.999,0,96,-0.199912\n0.0554401\n0.0901931\n-0.0963238\n-...,1280,720,/home/amos/programs/CineFace/data/videos/shini...
17,0.390,0.186,0.555,0.582,0.431,0.327,0.510,0.312,0.475,0.381,...,0.479,0.512,0.467,0.998,0,120,-0.150874\n0.0483781\n0.113282\n-0.137949\n-0....,1280,720,/home/amos/programs/CineFace/data/videos/shini...


In [118]:
encodings = shining['encoding'].map(parse_vector).tolist()
labels = dlib.chinese_whispers_clustering(encodings, 0.5)
shining = shining.assign(label=labels)
counts = shining['label'].value_counts()
df_cnt = shining.merge(counts,
              how='left',
              on='label')
top = df_cnt[df_cnt['count'] >= 15]
top.groupby('count').max()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,mouth_left_x,mouth_left_y,confidence,face_num,frame_num,encoding,img_width,img_height,filepath,label
count,,,,,,,,,,,,,,,,,,,,,
99,0.525,0.39,0.622,0.608,0.555,0.437,0.588,0.467,0.568,0.475,...,0.587,0.523,1.0,0,7944,-0.199912\n0.0554401\n0.0901931\n-0.0963238\n-...,1280,720,/home/amos/programs/CineFace/data/videos/shini...,0
180,0.652,0.40,0.761,0.669,0.671,0.517,0.722,0.500,0.692,0.589,...,0.723,0.607,1.0,0,8544,-0.207012\n0.125416\n0.0609809\n-0.0879595\n-0...,1280,720,/home/amos/programs/CineFace/data/videos/shini...,2


In [117]:
print(len(encodings))

303
